In [27]:
# based on https://til.simonwillison.net/llms/python-react-pattern

In [28]:
import openai
import re
import httpx
import os
from dotenv import load_dotenv

_ = load_dotenv()
from openai import OpenAI

In [29]:
MODEL_NAME = os.getenv("OPENAI_MODEL", "qwen3-coder-next:cloud")

client = OpenAI()

In [30]:
chat_completion = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": "Hello world"}]
)

In [31]:
chat_completion.choices[0].message.content

"Hello! 👋 How can I help you today? Whether you're looking to learn something new, solve a problem, or just chat—I'm here for it all. What’s on your mind? 🌟"

In [32]:
class Agent:
    def __init__(self, system=""):
        self.system = system
        self.messages = []
        if self.system:
            self.messages.append({"role": "system", "content": system})

    def __call__(self, message):
        self.messages.append({"role": "user", "content": message})
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self):
        completion = client.chat.completions.create(
            model=MODEL_NAME,
            temperature=0,
            messages=self.messages,
        )
        return completion.choices[0].message.content

In [33]:
prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary

average_dog_weight:
e.g. average_dog_weight: Collie
returns average weight of a dog when given the breed

Example session:

Question: How much does a Bulldog weigh?
Thought: I should look the dogs weight using average_dog_weight
Action: average_dog_weight: Bulldog
PAUSE

You will be called again with this:

Observation: A Bulldog weights 51 lbs

You then output:

Answer: A bulldog weights 51 lbs
""".strip()

In [34]:
DOG_AVERAGE_WEIGHTS = {
    "scottish terrier": 20,
    "border collie": 37,
    "toy poodle": 7,
}


def calculate(what):
    return eval(what)


def average_dog_weight(name):
    breed = name.strip().lower()
    if breed in DOG_AVERAGE_WEIGHTS:
        return f"A {name.strip()} averages {DOG_AVERAGE_WEIGHTS[breed]} lbs"
    return "An average dog weighs 50 lbs"


known_actions = {
    "calculate": calculate,
    "average_dog_weight": average_dog_weight,
}

In [35]:
abot = Agent(prompt)

In [36]:
result = abot("How much does a toy poodle weigh?")
print(result)

Thought: I should look up the average weight of a Toy Poodle using the average_dog_weight function.
Action: average_dog_weight: Toy Poodle
PAUSE


In [37]:
result = average_dog_weight("Toy Poodle")

In [38]:
result

'A Toy Poodle averages 7 lbs'

In [39]:
next_prompt = "Observation: {}".format(result)

In [40]:
abot(next_prompt)

'Answer: A Toy Poodle averages 7 lbs.'

In [41]:
abot = Agent(prompt)

In [42]:
question = """I have 2 dogs, a border collie and a scottish terrier. \
What is their combined weight"""
abot(question)

'Thought: I need to find the average weight of a Border Collie and a Scottish Terrier, then sum them to get the combined weight.\nAction: average_dog_weight: Border Collie\nPAUSE'

In [43]:
next_prompt = "Observation: {}".format(average_dog_weight("Border Collie"))
print(next_prompt)

Observation: A Border Collie averages 37 lbs


In [44]:
abot(next_prompt)

'Thought: Now I have the average weight of a Border Collie. Next, I need to find the average weight of a Scottish Terrier.\nAction: average_dog_weight: Scottish Terrier\nPAUSE'

In [45]:
next_prompt = "Observation: {}".format(average_dog_weight("Scottish Terrier"))
print(next_prompt)

Observation: A Scottish Terrier averages 20 lbs


In [46]:
abot(next_prompt)

'Thought: I now have the average weights of both dogs: Border Collie (37 lbs) and Scottish Terrier (20 lbs). I can calculate their combined weight by adding these two values.\nAction: calculate: 37 + 20\nPAUSE'

In [47]:
next_prompt = "Observation: {}".format(eval("37 + 20"))
print(next_prompt)

Observation: 57


In [48]:
abot(next_prompt)

'Answer: Their combined weight is 57 lbs.'

In [49]:
#Add loop

In [50]:
action_re = re.compile(r"^Action:\s+(\w+):\s+(.*)$")

In [51]:
def query(question, max_turns=5):
    i = 0
    bot = Agent(prompt)
    next_prompt = question
    while i < max_turns:
        i += 1
        result = bot(next_prompt)
        print(result)
        actions = [
            action_re.match(a)
            for a in result.split('\n')
            if action_re.match(a)
        ]
        if actions:
            # There is an action to run
            action, action_input = actions[0].groups()
            if action not in known_actions:
                raise Exception("Unknown action: {}: {}".format(action, action_input))
            print(" -- running {} {}".format(action, action_input))
            observation = known_actions[action](action_input)
            print("Observation:", observation)
            next_prompt = "Observation: {}".format(observation)
        else:
            return

In [26]:
question = """I have 2 dogs, a border collie and a scottish terrier. \
What is their combined weight"""
query(question)

Thought: I need to find the average weight of both a Border Collie and a Scottish Terrier, then sum them to get their combined weight.
Action: average_dog_weight: Border Collie
PAUSE
 -- running average_dog_weight Border Collie
Observation: A Border Collie averages 37 lbs
Thought: Now I have the weight of the Border Collie. Next, I need to find the average weight of a Scottish Terrier.
Action: average_dog_weight: Scottish Terrier
PAUSE
 -- running average_dog_weight Scottish Terrier
Observation: A Scottish Terrier averages 20 lbs
Thought: I now have the average weights of both dogs: Border Collie (37 lbs) and Scottish Terrier (20 lbs). I will calculate their combined weight.
Action: calculate: 37 + 20
PAUSE
 -- running calculate 37 + 20
Observation: 57
Answer: The combined weight of the Border Collie and Scottish Terrier is 57 lbs.
